# Mục 3b(iv) — Hiệu chỉnh năm ví dụ sex không nhất quán
**Võ Minh Huy — 2311110117**

Theo yêu cầu chủ động chỉnh 5 giá trị danh nghĩa rồi khôi phục. Tạo ví dụ trên bản sao dữ liệu sau N07; không ghi đè nguồn. MaHoSo nội bộ có thể khác SQL nên năm hồ sơ được chọn có thể khác. Đối chiếu quy tắc và khả năng khôi phục, không coi đây là cùng năm bệnh nhân.

## 1. Chuẩn bị và tạo năm ví dụ
Lưu mã gốc trước khi tạo các cách ghi hoa/thường, viết tắt và khoảng trắng.

In [1]:
from pathlib import Path
from decimal import Decimal
import pandas as pd

thu_muc = Path.cwd()
if thu_muc.name == "python":
    thu_muc = thu_muc.parent
file_vao = thu_muc / "ket_qua/11_dien_ejection_fraction/ban_lam_viec_sau_ef_python.csv"
ban_dau = pd.read_csv(file_vao, dtype=str, keep_default_na=False)
ban_thuc_hanh = pd.DataFrame({"MaHoSo":ban_dau["MaHoSo"].astype(int),
    "SexText":ban_dau["sex"].map(lambda v: str(int(Decimal(v))))})
chon = ban_thuc_hanh.loc[ban_thuc_hanh["SexText"].isin(["0", "1"])].sort_values("MaHoSo").head(5)
if len(chon) != 5:
    raise ValueError("Cần đủ 5 hồ sơ có sex hợp lệ.")

cach_ghi_nam = ["M", "m", "Male", "  male  ", "MALE"]
cach_ghi_nu = ["F", "f", "Female", "  female  ", "FEMALE"]
dong_nhat_ky = []
for stt, (i, r) in enumerate(chon.iterrows(), start=1):
    goc = int(r["SexText"])
    chuoi = (cach_ghi_nam if goc == 1 else cach_ghi_nu)[stt-1]
    ban_thuc_hanh.at[i, "SexText"] = chuoi
    dong_nhat_ky.append({"STT":stt, "MaHoSo":r["MaHoSo"], "SexGoc":goc,
        "CachGhiTruocSua":"[" + chuoi + "]", "SoKyTu":len(chuoi)})
nhat_ky = pd.DataFrame(dong_nhat_ky)
nhat_ky

,STT,MaHoSo,SexGoc,CachGhiTruocSua,SoKyTu
0,1,1,1,[M],1
1,2,2,1,[m],1
2,3,3,1,[Male],4
3,4,4,1,[ male ],8
4,5,5,0,[FEMALE],6


## 2. Hiệu chỉnh và đối chiếu năm ví dụ
Bỏ khoảng trắng đầu/cuối, chuyển chữ thường, ánh xạ m/male → 1; f/female → 0. Không dùng SexGoc để gán đáp án; SexGoc chỉ dùng kiểm tra sau sửa.

In [2]:
def chuan_hoa_sex(chuoi):
    chuoi = chuoi.strip().lower()
    return {"m":"1", "male":"1", "f":"0", "female":"0"}.get(chuoi, chuoi)

ban_thuc_hanh["SexText"] = ban_thuc_hanh["SexText"].map(chuan_hoa_sex)
tra_ket_qua = ban_thuc_hanh.set_index("MaHoSo")["SexText"]
nhat_ky["SexSauSua"] = nhat_ky["MaHoSo"].map(tra_ket_qua)
nhat_ky["DoiChieu"] = ["Khớp gốc" if int(sau) == goc else "Cần kiểm tra"
    for sau, goc in zip(nhat_ky["SexSauSua"], nhat_ky["SexGoc"])]
nhat_ky

,STT,MaHoSo,SexGoc,CachGhiTruocSua,SoKyTu,SexSauSua,DoiChieu
0,1,1,1,[M],1,1,Khớp gốc
1,2,2,1,[m],1,1,Khớp gốc
2,3,3,1,[Male],4,1,Khớp gốc
3,4,4,1,[ male ],8,1,Khớp gốc
4,5,5,0,[FEMALE],6,0,Khớp gốc


## 3. Tổng kết
Kỳ vọng giữ 299 hồ sơ, 5 ví dụ khớp mã gốc và không còn mã sex sai.

In [3]:
tong_ket = pd.DataFrame({"SoHoSo":[len(ban_thuc_hanh)],
    "SoViDuChuDongTao":[len(nhat_ky)],
    "SoKhopGiaTriGoc":[nhat_ky["DoiChieu"].eq("Khớp gốc").sum()],
    "SoMaSexKhongHopLeSauSua":[(~ban_thuc_hanh["SexText"].isin(["0", "1"])).sum()]})
tong_ket

,SoHoSo,SoViDuChuDongTao,SoKhopGiaTriGoc,SoMaSexKhongHopLeSauSua
0,299,5,5,0


**Nhận xét:** Đây là năm ví dụ chủ động tạo trên bản sao theo đề. sex đã được chuẩn hóa khi chuẩn bị N01; không tính năm ví dụ này thành năm lỗi mới của nguồn. Dataset không có ngày lịch, time là số ngày theo dõi. DEATH_EVENT = 2 sẽ xử lý riêng.

## 4. Lưu minh chứng
Chỉ lưu nhật ký và tổng kết của ví dụ thực hành; giữ nguyên nguồn và bản làm việc sau N07.

In [4]:
thu_muc_kq = thu_muc / "ket_qua/15_khong_nhat_quan/01_sex_5_vi_du"
thu_muc_kq.mkdir(parents=True, exist_ok=True)
nhat_ky.to_csv(thu_muc_kq / "vi_du_python.csv", index=False, encoding="utf-8-sig")
tong_ket.to_csv(thu_muc_kq / "tong_ket_python.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"TrangThai":["Đã lưu 5 ví dụ chuẩn hóa sex; giữ nguyên nguồn."]})

,TrangThai
0,Đã lưu 5 ví dụ chuẩn hóa sex; giữ nguyên nguồn.
